# 00 — Foundations Lab: Pricing, Premium Decomposition, and the Chain

This lab makes the lesson concrete. You will:

1. Price DEMO options across strikes and DTE with `pricing.bsm_price`.
2. **Decompose** each premium into intrinsic + extrinsic value and see the split.
3. Explore the real `DEMO` option chain: bid/ask, spreads, liquidity, moneyness.

Everything runs offline on the bundled sample chains. Spot for DEMO is **$100**, IV ~**25%**.

## Setup

Time is always in **years** in this library: 45 DTE is `45/365`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optionslab import pricing, data

SPOT = 100.0      # DEMO underlying
VOL  = 0.25       # ~25% implied vol
DTE  = 45
t    = DTE / 365  # convention: time in YEARS
round(t, 4)

## 1. Price a single option

Price the ATM 100 call at 45 DTE. Compare it to the chain's quoted mid later on.

In [ ]:
atm_call = pricing.bsm_price('call', SPOT, strike=100, t=t, vol=VOL)
atm_put  = pricing.bsm_price('put',  SPOT, strike=100, t=t, vol=VOL)
print(f'ATM 100 call: {atm_call:.2f}')
print(f'ATM 100 put : {atm_put:.2f}')

## 2. Price a call across strikes

Walk the strikes from deep ITM (80) to deep OTM (120). Watch the price fall smoothly.

In [ ]:
strikes = np.arange(80, 122.5, 2.5)
call_px = [pricing.bsm_price('call', SPOT, k, t, VOL) for k in strikes]
for k, p in zip(strikes, call_px):
    print(f'strike {k:6.1f}  call {p:6.2f}')

## 3. Decompose premium into intrinsic + extrinsic

The core skill from the lesson: `premium = intrinsic + extrinsic`.
Intrinsic for a call is `max(spot - strike, 0)`; extrinsic is whatever is left.

In [ ]:
def decompose_call(spot, strike, price):
    intrinsic = max(spot - strike, 0.0)
    extrinsic = price - intrinsic
    return intrinsic, extrinsic

for k in [80, 90, 100, 110, 120]:
    p = pricing.bsm_price('call', SPOT, k, t, VOL)
    intr, extr = decompose_call(SPOT, k, p)
    print(f'strike {k:5.1f}  price {p:6.2f}  intrinsic {intr:6.2f}  extrinsic {extr:6.2f}')

Notice the **ATM 100 call has the most extrinsic value** — the market charges the most for
uncertainty exactly where the outcome is most in doubt. Let's see that as a picture.

In [ ]:
prices    = np.array([pricing.bsm_price('call', SPOT, k, t, VOL) for k in strikes])
intrinsic = np.maximum(SPOT - strikes, 0.0)
extrinsic = prices - intrinsic
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(strikes, intrinsic, width=1.6, label='intrinsic')
ax.bar(strikes, extrinsic, width=1.6, bottom=intrinsic, label='extrinsic')
ax.axvline(SPOT, color='k', ls='--', lw=1, label='spot=100')
ax.set_xlabel('strike'); ax.set_ylabel('call premium'); ax.legend(); ax.set_title('DEMO 45-DTE call: intrinsic vs extrinsic')
plt.show()

## 4. Extrinsic value and the decay clock

Hold spot and strike fixed at ATM; shorten the time. Extrinsic value (all of the ATM premium)
shrinks — and shrinks *faster* as expiration nears.

In [ ]:
for dte in [90, 60, 45, 30, 21, 7, 1]:
    p = pricing.bsm_price('call', SPOT, 100, dte/365, VOL)
    print(f'{dte:3d} DTE   ATM call {p:5.2f}   (all extrinsic)')

## 5. Load and read the real DEMO chain

`data.load_sample_chain` returns a DataFrame with the documented columns. Inspect it.

In [ ]:
chain = data.load_sample_chain('DEMO')
print('available chains:', data.list_sample_chains())
print('spot:', chain['spot'].iloc[0])
chain.head()

## 6. Slice the 45-DTE calls and look at liquidity

Filter to 45-DTE calls, compute the bid-ask spread, and see how the market **thins in the wings**.

In [ ]:
c45 = chain[(chain.kind == 'call') & (chain.expiry_days == 45)].copy()
c45['spread'] = c45['ask'] - c45['bid']
c45['spread_pct'] = 100 * c45['spread'] / c45['mid'].where(c45['mid'] > 0)
cols = ['strike', 'bid', 'ask', 'mid', 'spread', 'spread_pct', 'open_interest', 'volume']
c45[cols].round(2).to_string(index=False)

The 100 call: penny-wide spread, OI > 8,000 — deeply tradeable. The far-OTM 130 call: spread as
wide as the option is worth, thin OI — a trap. **Liquidity is the filter you apply first.**

## 7. Model vs. market

Compare our flat-vol BSM price to the chain's quoted mid at each strike. They differ because the
market prices a *different IV per strike* (skew — module 02). The chain's `iv` column shows it.

In [ ]:
c45 = c45.sort_values('strike')
c45['model_flatvol'] = [pricing.bsm_price('call', SPOT, k, 45/365, 0.25) for k in c45.strike]
c45[['strike', 'mid', 'model_flatvol', 'iv']].round(3).to_string(index=False)

## Experiments

Change these and re-run — this is where the intuition sticks:

1. In section 3, switch `'call'` to `'put'` and rewrite `decompose_call` for puts
   (`intrinsic = max(strike - spot, 0)`). Which put strike now holds the most extrinsic value?
2. In section 4, raise `VOL` from 0.25 to 0.45 and re-run the decay table. How much more
   extrinsic value is there to decay when IV is high?
3. In section 6, change `expiry_days == 45` to `== 7` and `== 180`. How does the wing liquidity
   (OI, spreads) change with DTE?
4. Price the 90 call at 45 DTE, then at 7 DTE. How much of its value is intrinsic in each case,
   and why does the deep-ITM option barely decay?
5. Load the `HIGHVOL` chain (spot 62, IV ~55%) and repeat section 6. Are the ATM spreads tighter
   or wider in percentage terms than DEMO's?